In [ ]:
import pandas as pd
import openml

from openml_flow_minimal_cc18 import (
    load_cc18_from_openml,
    run_cc18_pipeline,
    run_standard_cc18_experiments,
    get_models,
    compute_permutation_importance,

In [ ]:
import joblib

try:
    flows = joblib.load("flows_cc18.joblib")
except:
    flows = openml.flows.list_flows(
        flow=used_flow_ids.tolist(),
        output_format="dict"
    )
    joblib.dump(flows, "flows_cc18.joblib")

In [ ]:
from openml_flow_minimal_cc18 import DiskCache

cache = DiskCache("minimal_cache_cc18")

bundle = load_cc18_from_openml(
    function="predictive_accuracy",
    suite_name="OpenML-CC18",
    cache=cache,
)

tasks_df = bundle["tasks_df"]
evals_df = bundle["evals_df"]

tasks_df.head(), evals_df.head()

In [ ]:
out = run_cc18_pipeline(
    flows=flows,
    tasks_df=tasks_df,
    evals_df=evals_df,
    agg_mode="mean",
    text_mode="tfidf",
    use_task_metafeatures=True,
    run_row_kfold=True,
    run_task_group_kfold=True,
    cv_folds=5,
    random_state=42,
    cache_dir="minimal_cache_cc18",
    results_dir="minimal_results_cc18",
)

In [ ]:
out["summary"]

In [ ]:
out["wilcoxon_row"]

In [ ]:
out["wilcoxon_task"]

In [ ]:
# standardní sada experimentů
all_out = run_standard_cc18_experiments(
    flows=flows,
    tasks_df=tasks_df,
    evals_df=evals_df,
    agg_modes=["mean", "max"],
    cv_folds=5,
    random_state=42,
    cache_dir="minimal_cache_cc18",
    results_dir="minimal_results_cc18",
)

In [ ]:
# spojení všech summary tabulek
all_summaries = []
for key, val in all_out.items():
    s = val["summary"].copy()
    s["experiment"] = key
    all_summaries.append(s)

summary_all = pd.concat(all_summaries, ignore_index=True)
summary_all.sort_values(["experiment", "split_mode", "r2_mean"], ascending=[True, True, False])

In [ ]:
# permutation importance pro nejlepší tree model v jednom konkrétním běhu
X = out["X"]
y = out["y"]
feature_names = out["feature_names"]

models = get_models(random_state=42)
model = models["extra_trees"]
model.fit(X, y)

perm_df = compute_permutation_importance(
    trained_model=model,
    X=X,
    y=y,
    feature_names=feature_names,
    n_repeats=10,
    random_state=42,
)

perm_df.head(20)

In [ ]:
perm_df.to_csv(out["run_dir"] / "permutation_importance_extra_trees.csv", index=False)